# Weight Initialization Techniques in Neural Networks
### A Comprehensive Guide

---

## Table of Contents

1. **Introduction** — Why Weight Initialization Matters
2. **The Vanishing & Exploding Gradient Problem** — Mathematical Foundation
3. **Zero Initialization** — The Anti-Pattern
4. **Random Initialization** — Breaking Symmetry
5. **Xavier/Glorot Initialization** — For Sigmoid & Tanh
6. **He/Kaiming Initialization** — For ReLU Networks
7. **LeCun Initialization** — For SELU Networks
8. **Orthogonal Initialization** — For RNNs and Deep Networks
9. **Sparse Initialization** — For Large Networks
10. **Comparative Analysis** — Visualizing Activations Across Methods
11. **Summary & Decision Guide**

## 1. Introduction — Why Weight Initialization Matters

Weight initialization is one of the most critical yet often overlooked aspects of training neural networks. The initial values assigned to a network's parameters determine:

* **Whether the network can learn at all** — poor initialization can make gradients vanish or explode from the very first forward pass
* **How fast the network converges** — good initialization places the network in a favorable region of the loss landscape
* **Whether the network finds a good local minimum** — initialization acts as an implicit regularizer

### The Core Problem

Consider a feedforward network with $$L$$ layers. The output of layer $$l$$ is:

$$z^{[l]} = W^{[l]} \cdot a^{[l-1]} + b^{[l]}$$

$$a^{[l]} = g(z^{[l]})$$

where $$W^{[l]}$$ is the weight matrix, $$b^{[l]}$$ is the bias vector, and $$g$$ is the activation function.

During backpropagation, the gradient of the loss with respect to weights in layer $$l$$ involves a product of Jacobians:

$$\frac{\partial \mathcal{L}}{\partial W^{[l]}} = \frac{\partial \mathcal{L}}{\partial a^{[L]}} \cdot \prod_{k=l+1}^{L} \frac{\partial a^{[k]}}{\partial a^{[k-1]}} \cdot \frac{\partial a^{[l]}}{\partial W^{[l]}}$$

This **product of Jacobians** is the root cause of training instability. If the spectral norm of each Jacobian is:
* **> 1**: Gradients grow exponentially → **Exploding Gradients**
* **< 1**: Gradients shrink exponentially → **Vanishing Gradients**
* **≈ 1**: Gradients remain stable → **Healthy Training**

The goal of proper weight initialization is to ensure that both activations (forward pass) and gradients (backward pass) maintain reasonable magnitudes across all layers.

## 2. The Vanishing & Exploding Gradient Problem

### Mathematical Analysis

Consider a deep linear network (no activation functions) with $$L$$ layers, each with weight matrix $$W^{[l]}$$. The network output is:

$$\hat{y} = W^{[L]} \cdot W^{[L-1]} \cdots W^{[1]} \cdot x$$

If all weight matrices are identical with singular values $$\sigma_i$$, after $$L$$ layers the effective singular values become $$\sigma_i^L$$.

### Variance Analysis

For a single neuron in layer $$l$$ with $$n_{in}$$ inputs:

$$z = \sum_{i=1}^{n_{in}} w_i \cdot x_i$$

Assuming $$w_i$$ and $$x_i$$ are independent with zero mean:

$$\text{Var}(z) = n_{in} \cdot \text{Var}(w) \cdot \text{Var}(x)$$

Across $$L$$ layers:

$$\text{Var}(a^{[L]}) = \text{Var}(x) \cdot \prod_{l=1}^{L} n_{l} \cdot \text{Var}(W^{[l]})$$

**Key Insight**: If $$n_l \cdot \text{Var}(W^{[l]}) > 1$$ for each layer, activations explode. If $$< 1$$, they vanish.

### The Ideal Condition

To maintain stable activations, we need:

$$n_l \cdot \text{Var}(W^{[l]}) = 1 \quad \Rightarrow \quad \text{Var}(W^{[l]}) = \frac{1}{n_l}$$

Similarly, for stable gradients during backpropagation:

$$n_{l+1} \cdot \text{Var}(W^{[l]}) = 1 \quad \Rightarrow \quad \text{Var}(W^{[l]}) = \frac{1}{n_{l+1}}$$

where $$n_l$$ is the fan-in (number of inputs) and $$n_{l+1}$$ is the fan-out (number of outputs).

In [0]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

def simulate_forward_pass(n_layers, n_neurons, init_std, activation='linear'):
    """Simulate forward pass and track activation statistics."""
    x = np.random.randn(1000, n_neurons)  # batch of 1000 samples
    means = []
    stds = []
    
    for l in range(n_layers):
        W = np.random.randn(n_neurons, n_neurons) * init_std
        x = x @ W
        if activation == 'tanh':
            x = np.tanh(x)
        elif activation == 'relu':
            x = np.maximum(0, x)
        means.append(np.mean(x))
        stds.append(np.std(x))
    
    return means, stds

# Compare different initialization scales
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Too small
means, stds = simulate_forward_pass(50, 256, 0.01, 'tanh')
axes[0].plot(stds, 'r-', linewidth=2)
axes[0].set_title('Too Small (std=0.01)\nVanishing Activations', fontsize=13)
axes[0].set_xlabel('Layer')
axes[0].set_ylabel('Std of Activations')
axes[0].set_ylim(0, 2)

# Too large
means, stds = simulate_forward_pass(50, 256, 1.0, 'tanh')
axes[1].plot(stds, 'r-', linewidth=2)
axes[1].set_title('Too Large (std=1.0)\nSaturation (tanh output ≈ ±1)', fontsize=13)
axes[1].set_xlabel('Layer')
axes[1].set_ylabel('Std of Activations')
axes[1].set_ylim(0, 2)

# Just right (Xavier)
xavier_std = np.sqrt(1.0 / 256)
means, stds = simulate_forward_pass(50, 256, xavier_std, 'tanh')
axes[2].plot(stds, 'g-', linewidth=2)
axes[2].set_title(f'Xavier (std={xavier_std:.4f})\nStable Activations', fontsize=13)
axes[2].set_xlabel('Layer')
axes[2].set_ylabel('Std of Activations')
axes[2].set_ylim(0, 2)

plt.tight_layout()
plt.suptitle('Effect of Weight Initialization on Activation Magnitudes (50-Layer Network, tanh)', 
             fontsize=14, y=1.02)
plt.show()

## 3. Zero Initialization

### Mathematical Formulation

$$W^{[l]} = \mathbf{0}, \quad b^{[l]} = \mathbf{0}$$

All weights and biases are set to zero.

### The Symmetry Problem

If all weights are initialized to zero (or any constant), then for every neuron in a layer:

$$z_j^{[l]} = \sum_{i} 0 \cdot a_i^{[l-1]} + 0 = 0$$

All neurons compute the **same output**, receive the **same gradient**, and update **identically**. This is called the **symmetry problem** — the network effectively becomes a single neuron per layer regardless of width.

### Pros
* Simple to implement
* Biases can safely be initialized to zero (only weights suffer from symmetry)

### Cons
* **Fatal flaw**: All neurons learn identical features (symmetry is never broken)
* Network capacity is completely wasted — a 1000-neuron layer behaves like 1 neuron
* Gradients are zero for all hidden layers → **no learning occurs**
* Not even useful as a baseline — the network literally cannot learn

### When to Use
* **Never for weights** in hidden layers
* **Acceptable for biases** — biases don't suffer from symmetry since weights are different
* Sometimes used for **residual connections** where the initial residual should be zero (e.g., LoRA adapters)

In [0]:
import torch
import torch.nn as nn
import torch.optim as optim

# Demonstrate the symmetry problem with zero initialization
def demonstrate_zero_init():
    """Show that zero-initialized networks cannot learn."""
    torch.manual_seed(42)
    
    # Create a simple 3-layer network
    model = nn.Sequential(
        nn.Linear(10, 50),
        nn.ReLU(),
        nn.Linear(50, 50),
        nn.ReLU(),
        nn.Linear(50, 1)
    )
    
    # Zero initialize all weights
    for param in model.parameters():
        nn.init.zeros_(param)
    
    # Generate dummy data
    X = torch.randn(100, 10)
    y = torch.randn(100, 1)
    
    optimizer = optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()
    
    losses = []
    for epoch in range(100):
        pred = model(X)
        loss = criterion(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    
    # Check if neurons learned different features
    with torch.no_grad():
        first_layer_weights = model[0].weight.data
        print("=== Zero Initialization Analysis ===")
        print(f"\nFirst layer weight matrix shape: {first_layer_weights.shape}")
        print(f"Are all rows identical? {torch.allclose(first_layer_weights[0], first_layer_weights[1], atol=1e-6)}")
        print(f"Max weight value after training: {first_layer_weights.abs().max():.8f}")
        print(f"\nLoss at epoch 1: {losses[0]:.4f}")
        print(f"Loss at epoch 100: {losses[-1]:.4f}")
        print(f"Loss decreased? {losses[-1] < losses[0] * 0.99}")
        print("\n❌ Zero initialization prevents learning - all neurons remain identical!")
    
    return losses

zero_losses = demonstrate_zero_init()

## 4. Random Initialization (Naive)

### Mathematical Formulation

**Normal distribution:**
$$W^{[l]}_{ij} \sim \mathcal{N}(0, \sigma^2)$$

**Uniform distribution:**
$$W^{[l]}_{ij} \sim \mathcal{U}(-a, a) \quad \text{where } \text{Var} = \frac{a^2}{3}$$

The key question is: **what should $$\sigma$$ (or $$a$$) be?**

### The Problem with Arbitrary Scale

If $$\sigma$$ is chosen without considering the network architecture:

* **Too small** (e.g., $$\sigma = 0.001$$): Activations shrink exponentially through layers
  $$\text{Var}(z^{[l]}) = n_{l-1} \cdot \sigma^2 \cdot \text{Var}(a^{[l-1]}) \to 0$$

* **Too large** (e.g., $$\sigma = 1.0$$): With tanh/sigmoid, neurons saturate; with ReLU, activations explode
  $$\text{Var}(z^{[l]}) = n_{l-1} \cdot \sigma^2 \cdot \text{Var}(a^{[l-1]}) \to \infty$$

### Pros
* **Breaks symmetry** — neurons learn different features
* Simple to implement and understand
* Works adequately for very shallow networks (1-2 hidden layers)

### Cons
* **Does not solve vanishing/exploding gradients** for deep networks
* Scale must be manually tuned per architecture
* No theoretical guarantees on training stability
* Performance degrades rapidly with depth

### When to Use
* Quick prototyping of shallow networks
* When combined with normalization layers (BatchNorm can partially compensate)
* As a **baseline** to compare against principled methods

In [0]:
def compare_random_scales(n_layers=30, n_neurons=256):
    """Compare different random initialization scales."""
    torch.manual_seed(42)
    
    scales = [0.001, 0.01, 0.1, 0.5, 1.0]
    fig, axes = plt.subplots(2, len(scales), figsize=(20, 8))
    
    for idx, scale in enumerate(scales):
        # Build network
        layers = []
        for _ in range(n_layers):
            layer = nn.Linear(n_neurons, n_neurons, bias=False)
            nn.init.normal_(layer.weight, mean=0, std=scale)
            layers.append(layer)
            layers.append(nn.Tanh())
        
        model = nn.Sequential(*layers)
        
        # Forward pass and collect activations
        x = torch.randn(500, n_neurons)
        activations = []
        with torch.no_grad():
            for i in range(0, len(layers), 2):  # step through linear layers
                x = layers[i](x)
                x = layers[i+1](x)
                activations.append(x.numpy().flatten())
        
        # Plot activation distribution at last layer
        axes[0, idx].hist(activations[-1], bins=50, density=True, alpha=0.7, color='steelblue')
        axes[0, idx].set_title(f'std={scale}\nFinal Layer Distribution', fontsize=11)
        axes[0, idx].set_xlim(-2, 2)
        
        # Plot std across layers
        layer_stds = [np.std(a) for a in activations]
        axes[1, idx].plot(layer_stds, 'b-', linewidth=2)
        axes[1, idx].set_title(f'Activation Std Across Layers', fontsize=11)
        axes[1, idx].set_xlabel('Layer')
        axes[1, idx].set_ylabel('Std')
        axes[1, idx].set_ylim(0, 1.5)
    
    plt.suptitle('Random Initialization: Effect of Scale on 30-Layer Tanh Network', fontsize=14)
    plt.tight_layout()
    plt.show()

compare_random_scales()

## 5. Xavier/Glorot Initialization (2010)

*Reference: Glorot & Bengio, "Understanding the difficulty of training deep feedforward neural networks", AISTATS 2010*

### Motivation

Xavier initialization derives the optimal variance by requiring that both the **forward pass activations** and **backward pass gradients** maintain the same variance across layers.

### Mathematical Derivation

**Forward pass condition** (preserve activation variance):
$$\text{Var}(a^{[l]}) = \text{Var}(a^{[l-1]}) \quad \Rightarrow \quad \text{Var}(W^{[l]}) = \frac{1}{n_{in}}$$

where $$n_{in}$$ is the number of input connections (fan-in).

**Backward pass condition** (preserve gradient variance):
$$\text{Var}\left(\frac{\partial \mathcal{L}}{\partial a^{[l-1]}}\right) = \text{Var}\left(\frac{\partial \mathcal{L}}{\partial a^{[l]}}\right) \quad \Rightarrow \quad \text{Var}(W^{[l]}) = \frac{1}{n_{out}}$$

where $$n_{out}$$ is the number of output connections (fan-out).

**Compromise (harmonic mean):**
$$\text{Var}(W^{[l]}) = \frac{2}{n_{in} + n_{out}}$$

### Initialization Formulas

**Xavier Normal:**
$$W^{[l]}_{ij} \sim \mathcal{N}\left(0, \frac{2}{n_{in} + n_{out}}\right)$$

**Xavier Uniform:**
$$W^{[l]}_{ij} \sim \mathcal{U}\left(-\sqrt{\frac{6}{n_{in} + n_{out}}}, \sqrt{\frac{6}{n_{in} + n_{out}}}\right)$$

The uniform bound comes from: $$\text{Var}(\mathcal{U}(-a,a)) = \frac{a^2}{3}$$, so $$\frac{a^2}{3} = \frac{2}{n_{in}+n_{out}}$$, giving $$a = \sqrt{\frac{6}{n_{in}+n_{out}}}$$.

### Assumptions
* Weights are initialized independently
* Inputs have zero mean
* Activation function is **linear around zero** (valid for sigmoid, tanh, but NOT ReLU)
* The key assumption: $$\mathbb{E}[g'(z)^2] \approx 1$$ where $$g$$ is the activation function

### Pros
* **Solves vanishing/exploding gradients** for networks with tanh/sigmoid activations
* Theoretically motivated with clear derivation
* Considers both forward and backward pass
* Works well for networks up to ~20 layers with appropriate activations
* Default in many frameworks (PyTorch Linear layers, TensorFlow)

### Cons
* **Assumes linear activation** — breaks down with ReLU (which zeros out half the inputs)
* With ReLU, variance halves at each layer: $$\text{Var}(\text{ReLU}(z)) = \frac{1}{2}\text{Var}(z)$$
* Not optimal for very deep networks (>50 layers) even with tanh
* Doesn't account for batch normalization or skip connections

### When to Use
* Networks with **sigmoid** or **tanh** activations
* **Autoencoders** (symmetric architecture makes fan-in/fan-out averaging natural)
* **Transformers** (attention layers use softmax, not ReLU)
* Moderate depth networks (5-20 layers)
* When using **GELU** or **Swish** activations (approximately linear near zero)

In [0]:
def demonstrate_xavier():
    """Demonstrate Xavier initialization with tanh activation."""
    torch.manual_seed(42)
    
    n_layers = 30
    n_neurons = 256
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # --- Xavier Normal ---
    layers_xn = []
    for _ in range(n_layers):
        layer = nn.Linear(n_neurons, n_neurons, bias=False)
        nn.init.xavier_normal_(layer.weight)
        layers_xn.append(layer)
    
    x = torch.randn(500, n_neurons)
    stds_xn = []
    with torch.no_grad():
        for layer in layers_xn:
            x = torch.tanh(layer(x))
            stds_xn.append(x.std().item())
    
    axes[0].plot(stds_xn, 'g-', linewidth=2, label='Xavier Normal')
    axes[0].axhline(y=stds_xn[0], color='gray', linestyle='--', alpha=0.5)
    axes[0].set_title('Xavier Normal + Tanh\nActivation Std per Layer', fontsize=12)
    axes[0].set_xlabel('Layer')
    axes[0].set_ylabel('Std of Activations')
    axes[0].legend()
    axes[0].set_ylim(0, 1)
    
    # --- Xavier Uniform ---
    layers_xu = []
    for _ in range(n_layers):
        layer = nn.Linear(n_neurons, n_neurons, bias=False)
        nn.init.xavier_uniform_(layer.weight)
        layers_xu.append(layer)
    
    x = torch.randn(500, n_neurons)
    stds_xu = []
    with torch.no_grad():
        for layer in layers_xu:
            x = torch.tanh(layer(x))
            stds_xu.append(x.std().item())
    
    axes[1].plot(stds_xu, 'b-', linewidth=2, label='Xavier Uniform')
    axes[1].axhline(y=stds_xu[0], color='gray', linestyle='--', alpha=0.5)
    axes[1].set_title('Xavier Uniform + Tanh\nActivation Std per Layer', fontsize=12)
    axes[1].set_xlabel('Layer')
    axes[1].set_ylabel('Std of Activations')
    axes[1].legend()
    axes[1].set_ylim(0, 1)
    
    # --- Xavier Normal + ReLU (showing the problem) ---
    layers_xr = []
    for _ in range(n_layers):
        layer = nn.Linear(n_neurons, n_neurons, bias=False)
        nn.init.xavier_normal_(layer.weight)
        layers_xr.append(layer)
    
    x = torch.randn(500, n_neurons)
    stds_xr = []
    with torch.no_grad():
        for layer in layers_xr:
            x = torch.relu(layer(x))
            stds_xr.append(x.std().item())
    
    axes[2].plot(stds_xr, 'r-', linewidth=2, label='Xavier Normal + ReLU')
    axes[2].set_title('Xavier Normal + ReLU\n⚠️ Activations Vanish!', fontsize=12)
    axes[2].set_xlabel('Layer')
    axes[2].set_ylabel('Std of Activations')
    axes[2].legend()
    axes[2].set_ylim(0, 1)
    
    plt.suptitle('Xavier/Glorot Initialization: Works with Tanh, Fails with ReLU', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Print the actual initialization values
    sample_layer = nn.Linear(256, 512, bias=False)
    nn.init.xavier_normal_(sample_layer.weight)
    fan_in, fan_out = 256, 512
    expected_std = np.sqrt(2.0 / (fan_in + fan_out))
    actual_std = sample_layer.weight.data.std().item()
    print(f"\n=== Xavier Normal for layer (256 → 512) ===")
    print(f"Expected std = sqrt(2 / (256 + 512)) = {expected_std:.6f}")
    print(f"Actual std = {actual_std:.6f}")

demonstrate_xavier()

## 6. He/Kaiming Initialization (2015)

*Reference: He et al., "Delving Deep into Rectifiers: Surpassing Human-Level Performance on ImageNet Classification", ICCV 2015*

### Motivation

Xavier initialization assumes the activation function is approximately linear around zero. ReLU violates this assumption because it **zeros out all negative inputs**, effectively halving the variance:

$$\text{Var}(\text{ReLU}(z)) = \frac{1}{2} \text{Var}(z) \quad \text{(for zero-mean } z \text{)}$$

This means with Xavier init and ReLU, variance decreases by a factor of $$\frac{1}{2}$$ at each layer, leading to vanishing activations.

### Mathematical Derivation

For a neuron with ReLU activation:
$$a = \text{ReLU}(z) = \text{ReLU}\left(\sum_{i=1}^{n_{in}} w_i x_i\right)$$

The variance of the output:
$$\text{Var}(a^{[l]}) = \frac{1}{2} \cdot n_{in} \cdot \text{Var}(W^{[l]}) \cdot \text{Var}(a^{[l-1]})$$

The factor $$\frac{1}{2}$$ comes from ReLU zeroing out negative values. To maintain variance:

$$\frac{1}{2} \cdot n_{in} \cdot \text{Var}(W^{[l]}) = 1 \quad \Rightarrow \quad \text{Var}(W^{[l]}) = \frac{2}{n_{in}}$$

### Initialization Formulas

**He Normal (fan-in mode):**
$$W^{[l]}_{ij} \sim \mathcal{N}\left(0, \frac{2}{n_{in}}\right)$$

**He Uniform (fan-in mode):**
$$W^{[l]}_{ij} \sim \mathcal{U}\left(-\sqrt{\frac{6}{n_{in}}}, \sqrt{\frac{6}{n_{in}}}\right)$$

**He Normal (fan-out mode):** *(for stable backpropagation)*
$$W^{[l]}_{ij} \sim \mathcal{N}\left(0, \frac{2}{n_{out}}\right)$$

### Variants for Leaky ReLU

For Leaky ReLU with negative slope $$\alpha$$:
$$\text{Var}(\text{LeakyReLU}_\alpha(z)) = \frac{1 + \alpha^2}{2} \cdot \text{Var}(z)$$

So the initialization becomes:
$$\text{Var}(W^{[l]}) = \frac{2}{(1 + \alpha^2) \cdot n_{in}}$$

For standard ReLU, $$\alpha = 0$$, recovering $$\frac{2}{n_{in}}$$.

### Pros
* **Solves vanishing/exploding gradients for ReLU networks**
* Enables training of very deep networks (100+ layers) like ResNets
* Theoretically principled — directly accounts for ReLU's variance reduction
* Supports variants (Leaky ReLU, PReLU, ELU) with the $$\alpha$$ parameter
* Industry standard for CNNs and most modern architectures

### Cons
* Assumes ReLU-family activations (not optimal for sigmoid/tanh)
* Forward-pass stability (fan-in) and backward-pass stability (fan-out) cannot both be satisfied simultaneously
* Does not account for skip connections (ResNets work despite this)
* Slight variance drift still accumulates over very deep networks (200+ layers)

### When to Use
* **Any network with ReLU, Leaky ReLU, PReLU, or ELU activations**
* **CNNs** (convolutional neural networks) — the standard choice
* **ResNets** and deep architectures
* **GANs** (Generative Adversarial Networks)
* Modern deep learning architectures in general (unless using transformers)

In [0]:
def demonstrate_he_init():
    """Demonstrate He initialization solves the ReLU problem."""
    torch.manual_seed(42)
    
    n_layers = 50
    n_neurons = 256
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # --- Xavier + ReLU (fails) ---
    x = torch.randn(500, n_neurons)
    stds_xavier = []
    with torch.no_grad():
        for _ in range(n_layers):
            layer = nn.Linear(n_neurons, n_neurons, bias=False)
            nn.init.xavier_normal_(layer.weight)
            x = torch.relu(layer(x))
            stds_xavier.append(x.std().item())
    
    axes[0, 0].plot(stds_xavier, 'r-', linewidth=2)
    axes[0, 0].set_title('Xavier + ReLU ❌\nActivations Vanish', fontsize=12)
    axes[0, 0].set_xlabel('Layer')
    axes[0, 0].set_ylabel('Activation Std')
    axes[0, 0].set_ylim(0, 1.5)
    
    # --- He + ReLU (works!) ---
    x = torch.randn(500, n_neurons)
    stds_he = []
    with torch.no_grad():
        for _ in range(n_layers):
            layer = nn.Linear(n_neurons, n_neurons, bias=False)
            nn.init.kaiming_normal_(layer.weight, mode='fan_in', nonlinearity='relu')
            x = torch.relu(layer(x))
            stds_he.append(x.std().item())
    
    axes[0, 1].plot(stds_he, 'g-', linewidth=2)
    axes[0, 1].set_title('He (Kaiming) + ReLU ✅\nStable Activations', fontsize=12)
    axes[0, 1].set_xlabel('Layer')
    axes[0, 1].set_ylabel('Activation Std')
    axes[0, 1].set_ylim(0, 1.5)
    
    # --- He + Leaky ReLU ---
    x = torch.randn(500, n_neurons)
    stds_leaky = []
    with torch.no_grad():
        for _ in range(n_layers):
            layer = nn.Linear(n_neurons, n_neurons, bias=False)
            nn.init.kaiming_normal_(layer.weight, mode='fan_in', nonlinearity='leaky_relu', a=0.2)
            x = torch.nn.functional.leaky_relu(x, negative_slope=0.2)
            stds_leaky.append(x.std().item())
    
    axes[1, 0].plot(stds_leaky, 'purple', linewidth=2)
    axes[1, 0].set_title('He + Leaky ReLU (slope=0.2) ✅', fontsize=12)
    axes[1, 0].set_xlabel('Layer')
    axes[1, 0].set_ylabel('Activation Std')
    axes[1, 0].set_ylim(0, 1.5)
    
    # --- Comparison of variance ---
    fan_in = 256
    he_var = 2.0 / fan_in
    xavier_var = 2.0 / (fan_in + fan_in)
    leaky_var = 2.0 / ((1 + 0.2**2) * fan_in)
    
    methods = ['Xavier\n(tanh)', 'He\n(ReLU)', 'He\n(LeakyReLU\nα=0.2)']
    variances = [xavier_var, he_var, leaky_var]
    stds_init = [np.sqrt(v) for v in variances]
    
    bars = axes[1, 1].bar(methods, stds_init, color=['steelblue', 'green', 'purple'], alpha=0.8)
    axes[1, 1].set_title('Initialization Std Comparison\n(fan_in = 256)', fontsize=12)
    axes[1, 1].set_ylabel('Initial Std of Weights')
    for bar, s in zip(bars, stds_init):
        axes[1, 1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.001,
                       f'{s:.4f}', ha='center', va='bottom', fontsize=10)
    
    plt.suptitle('He/Kaiming Initialization: Designed for ReLU Networks', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    print("\n=== He Initialization Formulas (fan_in=256) ===")
    print(f"He Normal std = sqrt(2/n_in) = sqrt(2/256) = {np.sqrt(2/256):.6f}")
    print(f"He Uniform bound = sqrt(6/n_in) = sqrt(6/256) = {np.sqrt(6/256):.6f}")
    print(f"Leaky ReLU (α=0.2) std = sqrt(2/((1+α²)*n_in)) = {np.sqrt(2/((1+0.04)*256)):.6f}")

demonstrate_he_init()

## 7. LeCun Initialization (1998)

*Reference: LeCun et al., "Efficient BackProp", Neural Networks: Tricks of the Trade, 1998*

### Motivation

LeCun initialization predates both Xavier and He. It was designed specifically for the **SELU (Scaled Exponential Linear Unit)** activation function, which has self-normalizing properties. It only considers the **forward pass** (fan-in only).

### Mathematical Formulation

**LeCun Normal:**
$$W^{[l]}_{ij} \sim \mathcal{N}\left(0, \frac{1}{n_{in}}\right)$$

**LeCun Uniform:**
$$W^{[l]}_{ij} \sim \mathcal{U}\left(-\sqrt{\frac{3}{n_{in}}}, \sqrt{\frac{3}{n_{in}}}\right)$$

### Relationship to Other Methods

| Method | Variance Formula | Factor |
|--------|-----------------|--------|
| LeCun  | $$\frac{1}{n_{in}}$$ | 1 |
| Xavier | $$\frac{2}{n_{in} + n_{out}}$$ | ≈1 (avg) |
| He     | $$\frac{2}{n_{in}}$$ | 2 |

LeCun = Xavier when $$n_{in} = n_{out}$$ (square weight matrices).

LeCun = He / 2 (half the variance of He).

### Self-Normalizing Neural Networks (SNNs)

When combined with SELU activation:
$$\text{SELU}(x) = \lambda \begin{cases} x & \text{if } x > 0 \\ \alpha(e^x - 1) & \text{if } x \leq 0 \end{cases}$$

where $$\lambda \approx 1.0507$$ and $$\alpha \approx 1.6733$$.

The LeCun init + SELU combination guarantees that activations converge to zero mean and unit variance, making the network **self-normalizing** without BatchNorm.

### Pros
* **Enables self-normalizing networks** with SELU — no BatchNorm needed
* Theoretically proven convergence to a fixed point (mean=0, var=1)
* Reduces architectural complexity (no normalization layers)
* Works well for fully-connected networks
* **Solves vanishing/exploding gradients** for self-normalizing architectures

### Cons
* **Only optimal with SELU activation** — suboptimal for ReLU or tanh
* Self-normalizing property requires strict architectural constraints (no skip connections, no pooling)
* Less studied than Xavier/He in modern architectures
* SELU itself is less popular than ReLU/GELU in practice
* Only considers fan-in (forward pass), ignores backward pass

### When to Use
* Networks with **SELU activation** (self-normalizing neural networks)
* **Fully-connected networks** without skip connections
* When you want to **avoid BatchNorm** for computational efficiency
* Tabular data models (MLPs for tabular data)
* Scenarios requiring **deterministic** forward passes (BatchNorm introduces mini-batch dependency)

In [0]:
def demonstrate_lecun():
    """Demonstrate LeCun initialization with SELU for self-normalizing networks."""
    torch.manual_seed(42)
    
    n_layers = 50
    n_neurons = 256
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # --- LeCun + SELU (self-normalizing) ---
    x = torch.randn(500, n_neurons)
    means_selu, stds_selu = [], []
    with torch.no_grad():
        for _ in range(n_layers):
            layer = nn.Linear(n_neurons, n_neurons, bias=False)
            # LeCun init: std = sqrt(1/fan_in)
            nn.init.normal_(layer.weight, mean=0, std=np.sqrt(1.0 / n_neurons))
            x = torch.nn.functional.selu(layer(x))
            means_selu.append(x.mean().item())
            stds_selu.append(x.std().item())
    
    axes[0, 0].plot(stds_selu, 'g-', linewidth=2, label='Std')
    axes[0, 0].plot(means_selu, 'b--', linewidth=1.5, label='Mean')
    axes[0, 0].axhline(y=1.0, color='gray', linestyle=':', alpha=0.5)
    axes[0, 0].axhline(y=0.0, color='gray', linestyle=':', alpha=0.5)
    axes[0, 0].set_title('LeCun + SELU ✅\nSelf-Normalizing', fontsize=12)
    axes[0, 0].set_xlabel('Layer')
    axes[0, 0].set_ylabel('Value')
    axes[0, 0].legend()
    axes[0, 0].set_ylim(-0.5, 2.0)
    
    # --- LeCun + ReLU (not ideal) ---
    x = torch.randn(500, n_neurons)
    stds_relu = []
    with torch.no_grad():
        for _ in range(n_layers):
            layer = nn.Linear(n_neurons, n_neurons, bias=False)
            nn.init.normal_(layer.weight, mean=0, std=np.sqrt(1.0 / n_neurons))
            x = torch.relu(layer(x))
            stds_relu.append(x.std().item())
    
    axes[0, 1].plot(stds_relu, 'r-', linewidth=2)
    axes[0, 1].set_title('LeCun + ReLU ❌\nActivations Vanish', fontsize=12)
    axes[0, 1].set_xlabel('Layer')
    axes[0, 1].set_ylabel('Activation Std')
    axes[0, 1].set_ylim(0, 1.5)
    
    # --- He + SELU (too large) ---
    x = torch.randn(500, n_neurons)
    stds_he_selu = []
    with torch.no_grad():
        for _ in range(n_layers):
            layer = nn.Linear(n_neurons, n_neurons, bias=False)
            nn.init.kaiming_normal_(layer.weight, mode='fan_in', nonlinearity='relu')
            x = torch.nn.functional.selu(layer(x))
            stds_he_selu.append(x.std().item())
    
    axes[1, 0].plot(stds_he_selu, 'orange', linewidth=2)
    axes[1, 0].set_title('He + SELU ⚠️\nOverscaled', fontsize=12)
    axes[1, 0].set_xlabel('Layer')
    axes[1, 0].set_ylabel('Activation Std')
    axes[1, 0].set_ylim(0, 3)
    
    # --- Activation distributions at layer 25 ---
    activations_data = {}
    for name, init_fn, act_fn in [
        ('LeCun+SELU', lambda w: nn.init.normal_(w, 0, np.sqrt(1.0/n_neurons)), torch.nn.functional.selu),
        ('He+ReLU', lambda w: nn.init.kaiming_normal_(w, mode='fan_in', nonlinearity='relu'), torch.relu),
    ]:
        x = torch.randn(1000, n_neurons)
        with torch.no_grad():
            for _ in range(25):
                layer = nn.Linear(n_neurons, n_neurons, bias=False)
                init_fn(layer.weight)
                x = act_fn(layer(x))
        activations_data[name] = x.numpy().flatten()
    
    axes[1, 1].hist(activations_data['LeCun+SELU'], bins=80, density=True, 
                    alpha=0.6, color='green', label='LeCun+SELU')
    axes[1, 1].hist(activations_data['He+ReLU'], bins=80, density=True, 
                    alpha=0.6, color='blue', label='He+ReLU')
    axes[1, 1].set_title('Activation Distribution at Layer 25', fontsize=12)
    axes[1, 1].set_xlabel('Activation Value')
    axes[1, 1].legend()
    axes[1, 1].set_xlim(-3, 3)
    
    plt.suptitle('LeCun Initialization: Designed for Self-Normalizing Networks (SELU)', fontsize=14)
    plt.tight_layout()
    plt.show()

demonstrate_lecun()

## 8. Orthogonal Initialization (2013)

*Reference: Saxe et al., "Exact solutions to the nonlinear dynamics of learning in deep linear networks", ICLR 2014*

### Motivation

Instead of controlling **variance**, orthogonal initialization controls the **singular values** of weight matrices. An orthogonal matrix $$W$$ satisfies:

$$W^T W = I \quad \text{(or } WW^T = I \text{ if } n_{out} < n_{in}\text{)}$$

All singular values of an orthogonal matrix are exactly 1, meaning the matrix preserves norms:

$$\|Wx\|_2 = \|x\|_2$$

This directly prevents both vanishing and exploding gradients because the gradient product:

$$\prod_{l} W^{[l]T}$$ has all singular values = 1

### Mathematical Formulation

**Algorithm:**
1. Generate a random matrix $$A \sim \mathcal{N}(0, 1)^{n_{out} \times n_{in}}$$
2. Compute the QR decomposition (or SVD): $$A = QR$$ (or $$A = U\Sigma V^T$$)
3. Set $$W = Q$$ (or $$U$$ if $$n_{out} \leq n_{in}$$, $$V^T$$ otherwise)
4. Optionally scale: $$W \leftarrow g \cdot W$$ where $$g$$ is a gain factor

**Gain factors for common activations:**

| Activation | Gain $$g$$ | Rationale |
|-----------|--------|----------|
| Linear/Identity | 1.0 | Preserves norm exactly |
| Tanh | $$\frac{5}{3} \approx 1.667$$ | Compensates for tanh's contraction |
| ReLU | $$\sqrt{2} \approx 1.414$$ | Compensates for zeroing half the values |
| SELU | $$\frac{3}{4}$$ | Matches SELU's self-normalizing property |

### Properties

* **Isometry**: Preserves the geometry of activation space (angles and distances)
* **No preferred direction**: All directions in input space are treated equally
* **Dynamical isometry**: In deep linear networks, enables information to flow without distortion

### Pros
* **Strongest guarantee against vanishing/exploding gradients** in linear networks
* Enables training of extremely deep networks (1000+ layers in theory)
* **Excellent for RNNs** — prevents gradient decay over long sequences
* Preserves geometric structure of data through layers
* No dependence on fan-in/fan-out (works for any layer shape)
* **Provably optimal** for deep linear networks (fastest convergence)

### Cons
* **Computationally expensive**: QR/SVD decomposition is $$O(n^3)$$ vs $$O(n^2)$$ for sampling
* Only exact orthogonality for square matrices — approximate for non-square
* Theoretical guarantees rely on **linear** networks; nonlinearities break exact orthogonality after one step
* Does not account for specific activation function properties (hence the gain factor heuristic)
* Overhead is negligible relative to training time, but noticeable at initialization

### When to Use
* **RNNs / LSTMs / GRUs** — the recurrent weight matrix benefits enormously
* Very deep networks (50+ layers) without skip connections
* **Physics-informed neural networks** (preserving geometric structure)
* When training is unstable with He/Xavier
* Networks requiring **exact gradient preservation** (e.g., normalizing flows)
* **GANs** where training stability is critical

In [0]:
def demonstrate_orthogonal():
    """Demonstrate orthogonal initialization for deep networks and RNNs."""
    torch.manual_seed(42)
    
    n_layers = 100  # Very deep!
    n_neurons = 256
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # --- Orthogonal + Linear (perfect preservation) ---
    x = torch.randn(500, n_neurons)
    norms_ortho = [x.norm(dim=1).mean().item()]
    with torch.no_grad():
        for _ in range(n_layers):
            layer = nn.Linear(n_neurons, n_neurons, bias=False)
            nn.init.orthogonal_(layer.weight, gain=1.0)
            x = layer(x)
            norms_ortho.append(x.norm(dim=1).mean().item())
    
    axes[0, 0].plot(norms_ortho, 'g-', linewidth=2)
    axes[0, 0].axhline(y=norms_ortho[0], color='gray', linestyle='--', alpha=0.5)
    axes[0, 0].set_title('Orthogonal + Linear\nPerfect Norm Preservation (100 layers)', fontsize=11)
    axes[0, 0].set_xlabel('Layer')
    axes[0, 0].set_ylabel('Mean L2 Norm')
    
    # --- Random Normal + Linear (explodes) ---
    x = torch.randn(500, n_neurons)
    norms_random = [x.norm(dim=1).mean().item()]
    with torch.no_grad():
        for _ in range(n_layers):
            layer = nn.Linear(n_neurons, n_neurons, bias=False)
            nn.init.normal_(layer.weight, std=1.0/np.sqrt(n_neurons))
            x = layer(x)
            norms_random.append(x.norm(dim=1).mean().item())
    
    axes[0, 1].plot(norms_random, 'r-', linewidth=2)
    axes[0, 1].set_title('Random Normal + Linear\nNorm Drifts Over 100 Layers', fontsize=11)
    axes[0, 1].set_xlabel('Layer')
    axes[0, 1].set_ylabel('Mean L2 Norm')
    
    # --- RNN simulation: gradient flow over time steps ---
    def simulate_rnn_gradient(init_type, n_steps=200):
        hidden_size = 128
        W_hh = torch.zeros(hidden_size, hidden_size)
        
        if init_type == 'orthogonal':
            nn.init.orthogonal_(W_hh)
        elif init_type == 'xavier':
            nn.init.xavier_normal_(W_hh)
        elif init_type == 'random':
            nn.init.normal_(W_hh, std=0.1)
        
        # Simulate gradient backprop through time
        gradient = torch.ones(hidden_size)
        grad_norms = [gradient.norm().item()]
        
        for t in range(n_steps):
            gradient = W_hh.T @ gradient * 0.9  # tanh derivative ≈ 0.9 average
            grad_norms.append(gradient.norm().item())
        
        return grad_norms
    
    for init_type, color, label in [
        ('orthogonal', 'green', 'Orthogonal'),
        ('xavier', 'blue', 'Xavier'),
        ('random', 'red', 'Random (std=0.1)'),
    ]:
        grad_norms = simulate_rnn_gradient(init_type)
        axes[1, 0].semilogy(grad_norms, color=color, linewidth=2, label=label)
    
    axes[1, 0].set_title('RNN Gradient Flow Over Time Steps\n(simulated BPTT)', fontsize=11)
    axes[1, 0].set_xlabel('Time Steps Back')
    axes[1, 0].set_ylabel('Gradient Norm (log scale)')
    axes[1, 0].legend()
    axes[1, 0].set_ylim(1e-10, 1e10)
    
    # --- Verify orthogonality ---
    layer = nn.Linear(256, 256, bias=False)
    nn.init.orthogonal_(layer.weight)
    W = layer.weight.data
    WtW = W.T @ W
    identity = torch.eye(256)
    error = (WtW - identity).abs().max().item()
    
    singular_values = torch.svd(W).S
    axes[1, 1].hist(singular_values.numpy(), bins=30, color='green', alpha=0.7, edgecolor='black')
    axes[1, 1].axvline(x=1.0, color='red', linestyle='--', linewidth=2, label='Target = 1.0')
    axes[1, 1].set_title(f'Singular Values of Orthogonal Matrix\nMax deviation from I: {error:.2e}', fontsize=11)
    axes[1, 1].set_xlabel('Singular Value')
    axes[1, 1].set_ylabel('Count')
    axes[1, 1].legend()
    
    plt.suptitle('Orthogonal Initialization: Norm-Preserving for Deep & Recurrent Networks', fontsize=14)
    plt.tight_layout()
    plt.show()

demonstrate_orthogonal()

## 9. Sparse Initialization (2013)

*Reference: Martens, "Deep learning via Hessian-free optimization", ICML 2010; Sutskever et al., "On the importance of initialization and momentum in deep learning", ICML 2013*

### Motivation

Sparse initialization creates weight matrices where most elements are zero, and only a fixed number of non-zero connections exist per neuron. This mimics the sparse connectivity found in biological neural networks and can improve gradient flow in specific architectures.

### Mathematical Formulation

For each neuron (row of the weight matrix):
1. Set all weights to 0
2. Randomly select $$k$$ connections (typically $$k = \lceil \sqrt{n_{in}} \rceil$$ or a fixed small number)
3. Initialize the $$k$$ non-zero weights from $$\mathcal{N}(0, \sigma^2)$$

The effective variance per neuron:
$$\text{Var}(z) = k \cdot \sigma^2 \cdot \text{Var}(x)$$

To maintain unit variance: $$\sigma = \frac{1}{\sqrt{k}}$$

### Sparsity Pattern

The weight matrix $$W \in \mathbb{R}^{n_{out} \times n_{in}}$$ has:
* Exactly $$k$$ non-zero entries per row
* Sparsity ratio: $$1 - \frac{k}{n_{in}}$$
* For large layers ($$n_{in} = 1024, k = 15$$): sparsity ≈ 98.5%

### Pros
* **Reduces effective fan-in** — each neuron sees fewer inputs, reducing variance issues
* Creates **diverse receptive fields** from the start
* **Memory efficient** for very large layers (sparse matrix storage)
* Can improve gradient flow by reducing interference between neurons
* **Prevents co-adaptation** early in training (similar to dropout's effect)
* Empirically works well for certain deep architectures

### Cons
* **Not suitable for CNNs** (kernel sizes are already small)
* The sparse structure is quickly lost during training (dense updates fill in zeros)
* Harder to implement efficiently on GPUs (sparse operations have overhead)
* No strong theoretical guarantees compared to Xavier/He
* **Can create dead neurons** if the sparse connections happen to receive zero gradients
* Hyperparameter $$k$$ requires tuning

### When to Use
* Very large **fully-connected layers** (1000+ neurons)
* **Second-order optimization** methods (Hessian-free, K-FAC)
* Networks where diversity of learned features is critical
* When combined with **sparse training** techniques (lottery ticket hypothesis)
* **Mixture of Experts** architectures (naturally sparse routing)

In [0]:
def demonstrate_sparse_init():
    """Demonstrate sparse initialization."""
    torch.manual_seed(42)
    
    def sparse_init(weight, sparsity=0.9, std=None):
        """Custom sparse initialization."""
        n_out, n_in = weight.shape
        nn.init.zeros_(weight)
        
        k = int(n_in * (1 - sparsity))  # non-zero connections per neuron
        if std is None:
            std = 1.0 / np.sqrt(k)
        
        for i in range(n_out):
            indices = torch.randperm(n_in)[:k]
            weight.data[i, indices] = torch.randn(k) * std
        
        return k
    
    n_neurons = 256
    n_layers = 30
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # --- Visualize sparse weight matrix ---
    W = torch.zeros(64, 64)
    sparse_init(W, sparsity=0.9)
    axes[0, 0].imshow(W.numpy() != 0, cmap='binary', aspect='auto')
    axes[0, 0].set_title('Sparse Weight Matrix (90% sparse)\nWhite = Non-zero', fontsize=11)
    axes[0, 0].set_xlabel('Input neuron')
    axes[0, 0].set_ylabel('Output neuron')
    
    # --- Compare activation flow ---
    results = {}
    for label, init_fn, color in [
        ('Sparse (90%)', lambda w: sparse_init(w, 0.9), 'green'),
        ('Sparse (95%)', lambda w: sparse_init(w, 0.95), 'blue'),
        ('Dense Xavier', lambda w: nn.init.xavier_normal_(w), 'orange'),
    ]:
        x = torch.randn(500, n_neurons)
        stds = []
        with torch.no_grad():
            for _ in range(n_layers):
                layer = nn.Linear(n_neurons, n_neurons, bias=False)
                init_fn(layer.weight)
                x = torch.tanh(layer(x))
                stds.append(x.std().item())
        results[label] = stds
        axes[0, 1].plot(stds, color=color, linewidth=2, label=label)
    
    axes[0, 1].set_title('Activation Std Across Layers (Tanh)', fontsize=11)
    axes[0, 1].set_xlabel('Layer')
    axes[0, 1].set_ylabel('Std')
    axes[0, 1].legend()
    axes[0, 1].set_ylim(0, 1)
    
    # --- Weight distribution comparison ---
    W_sparse = torch.zeros(256, 256)
    sparse_init(W_sparse, sparsity=0.9)
    W_dense = torch.zeros(256, 256)
    nn.init.xavier_normal_(W_dense)
    
    axes[1, 0].hist(W_sparse.numpy().flatten(), bins=80, density=True, 
                    alpha=0.6, color='green', label='Sparse (90%)')
    axes[1, 0].hist(W_dense.numpy().flatten(), bins=80, density=True, 
                    alpha=0.6, color='orange', label='Dense Xavier')
    axes[1, 0].set_title('Weight Distribution Comparison', fontsize=11)
    axes[1, 0].set_xlabel('Weight Value')
    axes[1, 0].legend()
    
    # --- Sparsity vs performance tradeoff ---
    sparsity_levels = [0.0, 0.5, 0.7, 0.8, 0.9, 0.95, 0.99]
    final_stds = []
    
    for sp in sparsity_levels:
        x = torch.randn(500, n_neurons)
        with torch.no_grad():
            for _ in range(20):
                layer = nn.Linear(n_neurons, n_neurons, bias=False)
                if sp == 0:
                    nn.init.xavier_normal_(layer.weight)
                else:
                    sparse_init(layer.weight, sparsity=sp)
                x = torch.tanh(layer(x))
        final_stds.append(x.std().item())
    
    axes[1, 1].bar([f'{s*100:.0f}%' for s in sparsity_levels], final_stds, 
                   color='steelblue', alpha=0.8)
    axes[1, 1].set_title('Final Activation Std vs Sparsity\n(After 20 Layers)', fontsize=11)
    axes[1, 1].set_xlabel('Sparsity Level')
    axes[1, 1].set_ylabel('Activation Std')
    axes[1, 1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Ideal range')
    axes[1, 1].legend()
    
    plt.suptitle('Sparse Initialization: Controlled Connectivity', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    print("\n=== Sparse Init Statistics (256x256 matrix, 90% sparse) ===")
    print(f"Non-zero entries per neuron: {int(256 * 0.1)} out of 256")
    print(f"Weight std for non-zero entries: 1/sqrt(k) = 1/sqrt({int(256*0.1)}) = {1/np.sqrt(int(256*0.1)):.4f}")
    print(f"Total parameters: {256*256:,}")
    print(f"Non-zero parameters: {int(256*256*0.1):,}")
    print(f"Memory savings: {90:.1f}%")

demonstrate_sparse_init()

## 10. Comprehensive Comparative Analysis

Let's now compare all initialization methods head-to-head in a realistic training scenario — training a multi-layer network on actual data and measuring convergence speed, final loss, and gradient health.

In [0]:
from torch.utils.data import DataLoader, TensorDataset

def full_training_comparison():
    """Train networks with different initializations on a classification task."""
    torch.manual_seed(42)
    np.random.seed(42)
    
    # Generate a non-trivial classification dataset
    n_samples = 2000
    n_features = 20
    n_classes = 5
    
    # Create clustered data
    X = []
    y = []
    for c in range(n_classes):
        center = np.random.randn(n_features) * 3
        samples = center + np.random.randn(n_samples // n_classes, n_features) * 0.8
        X.append(samples)
        y.extend([c] * (n_samples // n_classes))
    
    X = torch.FloatTensor(np.vstack(X))
    y = torch.LongTensor(y)
    
    dataset = TensorDataset(X, y)
    loader = DataLoader(dataset, batch_size=128, shuffle=True)
    
    # Define network architecture
    def create_network(init_name):
        model = nn.Sequential(
            nn.Linear(20, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 5)
        )
        
        # Apply initialization
        for m in model.modules():
            if isinstance(m, nn.Linear):
                if init_name == 'zeros':
                    nn.init.zeros_(m.weight)
                elif init_name == 'random_small':
                    nn.init.normal_(m.weight, std=0.01)
                elif init_name == 'random_large':
                    nn.init.normal_(m.weight, std=1.0)
                elif init_name == 'xavier_normal':
                    nn.init.xavier_normal_(m.weight)
                elif init_name == 'he_normal':
                    nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                elif init_name == 'orthogonal':
                    nn.init.orthogonal_(m.weight, gain=np.sqrt(2))  # ReLU gain
                
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
        
        return model
    
    # Training loop
    init_methods = {
        'Zeros': 'zeros',
        'Random (std=0.01)': 'random_small',
        'Random (std=1.0)': 'random_large',
        'Xavier Normal': 'xavier_normal',
        'He Normal': 'he_normal',
        'Orthogonal': 'orthogonal',
    }
    
    results = {}
    n_epochs = 50
    
    for display_name, init_name in init_methods.items():
        torch.manual_seed(42)
        model = create_network(init_name)
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.CrossEntropyLoss()
        
        epoch_losses = []
        epoch_accs = []
        grad_norms = []
        
        for epoch in range(n_epochs):
            total_loss = 0
            correct = 0
            total = 0
            batch_grad_norms = []
            
            for batch_X, batch_y in loader:
                optimizer.zero_grad()
                output = model(batch_X)
                loss = criterion(output, batch_y)
                loss.backward()
                
                # Track gradient norm
                total_norm = 0
                for p in model.parameters():
                    if p.grad is not None:
                        total_norm += p.grad.data.norm(2).item() ** 2
                batch_grad_norms.append(np.sqrt(total_norm))
                
                optimizer.step()
                
                total_loss += loss.item()
                _, predicted = output.max(1)
                correct += predicted.eq(batch_y).sum().item()
                total += batch_y.size(0)
            
            epoch_losses.append(total_loss / len(loader))
            epoch_accs.append(100. * correct / total)
            grad_norms.append(np.mean(batch_grad_norms))
        
        results[display_name] = {
            'losses': epoch_losses,
            'accs': epoch_accs,
            'grad_norms': grad_norms
        }
    
    # Plotting
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    colors = ['red', 'orange', 'purple', 'blue', 'green', 'darkgreen']
    
    for (name, data), color in zip(results.items(), colors):
        axes[0].plot(data['losses'], color=color, linewidth=2, label=name)
        axes[1].plot(data['accs'], color=color, linewidth=2, label=name)
        axes[2].semilogy(data['grad_norms'], color=color, linewidth=2, label=name)
    
    axes[0].set_title('Training Loss', fontsize=13)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Cross-Entropy Loss')
    axes[0].legend(fontsize=9)
    
    axes[1].set_title('Training Accuracy', fontsize=13)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend(fontsize=9)
    
    axes[2].set_title('Gradient Norm (log scale)', fontsize=13)
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('L2 Norm of Gradients')
    axes[2].legend(fontsize=9)
    
    plt.suptitle('Full Training Comparison: 6-Layer ReLU Network on Classification Task', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Summary table
    print("\n" + "="*75)
    print(f"{'Method':<20} {'Final Loss':>12} {'Final Acc':>12} {'Converged (epoch)':>18}")
    print("="*75)
    for name, data in results.items():
        final_loss = data['losses'][-1]
        final_acc = data['accs'][-1]
        # Find epoch where accuracy first exceeded 90%
        converged = next((i for i, a in enumerate(data['accs']) if a > 90), -1)
        conv_str = str(converged) if converged >= 0 else "Never"
        print(f"{name:<20} {final_loss:>12.4f} {final_acc:>11.1f}% {conv_str:>18}")
    print("="*75)

full_training_comparison()

In [0]:
def visualize_activation_distributions():
    """Visualize how different initializations affect activation distributions layer by layer."""
    torch.manual_seed(42)
    
    n_neurons = 256
    layers_to_show = [0, 4, 9, 19, 29]  # Layers 1, 5, 10, 20, 30
    
    init_configs = [
        ('Xavier Normal + Tanh', 'xavier_normal_', {}, torch.tanh),
        ('He Normal + ReLU', 'kaiming_normal_', {'nonlinearity': 'relu'}, torch.relu),
        ('Orthogonal + ReLU', 'orthogonal_', {'gain': np.sqrt(2)}, torch.relu),
    ]
    
    fig, axes = plt.subplots(len(init_configs), len(layers_to_show), figsize=(20, 12))
    
    for row, (name, init_fn_name, init_kwargs, act_fn) in enumerate(init_configs):
        x = torch.randn(2000, n_neurons)
        layer_idx = 0
        col = 0
        
        for i in range(30):
            layer = nn.Linear(n_neurons, n_neurons, bias=False)
            init_fn = getattr(nn.init, init_fn_name)
            init_fn(layer.weight, **init_kwargs)
            
            with torch.no_grad():
                x = act_fn(layer(x))
            
            if i in layers_to_show:
                data = x.numpy().flatten()
                axes[row, col].hist(data, bins=80, density=True, alpha=0.7, 
                                   color=['steelblue', 'green', 'purple'][row])
                axes[row, col].set_xlim(-3, 3)
                axes[row, col].set_title(f'Layer {i+1}\nstd={np.std(data):.3f}', fontsize=10)
                if col == 0:
                    axes[row, col].set_ylabel(name, fontsize=11, fontweight='bold')
                col += 1
    
    plt.suptitle('Activation Distributions Across Layers for Different Init Methods', fontsize=14)
    plt.tight_layout()
    plt.show()

visualize_activation_distributions()

## 11. Summary & Decision Guide

### Quick Reference Table

| Method | Variance | Activation | Solves Vanishing Grad? | Solves Exploding Grad? | Best For |
|--------|----------|------------|:---:|:---:|----------|
| **Zero** | 0 | Any | ❌ | ❌ | Never (except biases) |
| **Random** | Manual $$\sigma^2$$ | Any | ❌ | ❌ | Quick prototypes |
| **Xavier Normal** | $$\frac{2}{n_{in}+n_{out}}$$ | Sigmoid, Tanh | ✅ | ✅ | Transformers, Autoencoders |
| **Xavier Uniform** | $$\frac{2}{n_{in}+n_{out}}$$ | Sigmoid, Tanh | ✅ | ✅ | Same as Xavier Normal |
| **He Normal** | $$\frac{2}{n_{in}}$$ | ReLU, Leaky ReLU | ✅ | ✅ | CNNs, ResNets, GANs |
| **He Uniform** | $$\frac{2}{n_{in}}$$ | ReLU, Leaky ReLU | ✅ | ✅ | Same as He Normal |
| **LeCun** | $$\frac{1}{n_{in}}$$ | SELU | ✅ | ✅ | Self-normalizing nets |
| **Orthogonal** | N/A (SVs=1) | Any (with gain) | ✅ | ✅ | RNNs, very deep nets |
| **Sparse** | $$\frac{1}{k}$$ | Any | Partial | Partial | Large FC layers |

### Decision Flowchart

```
Start
  │
  ├─ Using ReLU/Leaky ReLU/PReLU/ELU?
  │     └─ YES → He (Kaiming) Initialization
  │
  ├─ Using Sigmoid/Tanh/GELU/Swish?
  │     └─ YES → Xavier (Glorot) Initialization
  │
  ├─ Using SELU (Self-Normalizing Network)?
  │     └─ YES → LeCun Initialization
  │
  ├─ Building an RNN/LSTM/GRU?
  │     └─ YES → Orthogonal Initialization (for recurrent weights)
  │
  ├─ Very deep (100+ layers) without BatchNorm?
  │     └─ YES → Orthogonal or LSUV
  │
  ├─ Transformer architecture?
  │     └─ YES → Xavier (scaled by 1/sqrt(2L) for deep transformers)
  │
  └─ Default → He Normal (most modern networks use ReLU variants)
```

### Modern Best Practices (2024+)

1. **For CNNs**: He Normal + ReLU (industry standard since ResNet)
2. **For Transformers**: Xavier or scaled initialization (GPT uses $$\frac{1}{\sqrt{2L}}$$ scaling)
3. **For RNNs**: Orthogonal for recurrent weights, He/Xavier for input weights
4. **For GANs**: He Normal or Orthogonal (training stability is paramount)
5. **For Transfer Learning**: Pre-trained weights (initialization becomes irrelevant for the backbone)
6. **With BatchNorm**: Any reasonable init works (BatchNorm compensates), but He is still preferred
7. **For Residual Networks**: He init + zero-initialize the final BN in each residual branch

### Key Takeaways

* Weight initialization is **not** a hyperparameter to tune — it's a **design choice** dictated by your activation function
* The "right" initialization keeps both activations and gradients at **unit scale** ($$O(1)$$)
* Modern architectures (with BatchNorm, skip connections, layer norm) are less sensitive to initialization, but starting well still speeds up convergence
* When in doubt: **He Normal for ReLU, Xavier Normal for everything else**

In [0]:
# ============================================================
# COMPLETE IMPLEMENTATION REFERENCE
# Weight Initialization in PyTorch and TensorFlow/Keras
# ============================================================

import torch
import torch.nn as nn
import numpy as np

print("=" * 70)
print("WEIGHT INITIALIZATION: COMPLETE PyTorch REFERENCE")
print("=" * 70)

# -----------------------------------------------------------
# 1. PYTORCH IMPLEMENTATIONS
# -----------------------------------------------------------

class WellInitializedNetwork(nn.Module):
    """Example of a properly initialized deep network."""
    
    def __init__(self, input_dim, hidden_dim, output_dim, n_hidden_layers=5, 
                 activation='relu', init_method='auto'):
        super().__init__()
        
        layers = []
        dims = [input_dim] + [hidden_dim] * n_hidden_layers + [output_dim]
        
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i+1]))
            if i < len(dims) - 2:  # No activation on last layer
                if activation == 'relu':
                    layers.append(nn.ReLU())
                elif activation == 'tanh':
                    layers.append(nn.Tanh())
                elif activation == 'selu':
                    layers.append(nn.SELU())
                elif activation == 'leaky_relu':
                    layers.append(nn.LeakyReLU(0.2))
        
        self.network = nn.Sequential(*layers)
        self._initialize_weights(activation, init_method)
    
    def _initialize_weights(self, activation, init_method):
        """Apply appropriate initialization based on activation function."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                if init_method == 'auto':
                    # Automatically choose based on activation
                    if activation in ['relu', 'leaky_relu']:
                        nn.init.kaiming_normal_(m.weight, 
                                              mode='fan_in',
                                              nonlinearity=activation if activation == 'leaky_relu' else 'relu')
                    elif activation in ['tanh', 'sigmoid']:
                        nn.init.xavier_normal_(m.weight)
                    elif activation == 'selu':
                        # LeCun initialization
                        fan_in = m.weight.size(1)
                        nn.init.normal_(m.weight, std=1.0 / np.sqrt(fan_in))
                    else:
                        nn.init.xavier_normal_(m.weight)
                elif init_method == 'orthogonal':
                    gain = nn.init.calculate_gain(activation if activation != 'selu' else 'linear')
                    nn.init.orthogonal_(m.weight, gain=gain)
                
                # Biases: always zero
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x):
        return self.network(x)


# Demonstrate usage
print("\n--- Creating networks with different configurations ---")

configs = [
    ('ReLU + He (auto)', 'relu', 'auto'),
    ('Tanh + Xavier (auto)', 'tanh', 'auto'),
    ('SELU + LeCun (auto)', 'selu', 'auto'),
    ('ReLU + Orthogonal', 'relu', 'orthogonal'),
]

for desc, act, init in configs:
    model = WellInitializedNetwork(784, 256, 10, n_hidden_layers=5, 
                                   activation=act, init_method=init)
    
    # Check weight statistics
    first_weight = list(model.parameters())[0]
    print(f"\n{desc}:")
    print(f"  First layer weight: mean={first_weight.data.mean():.6f}, "
          f"std={first_weight.data.std():.6f}")
    
    # Quick forward pass check
    x = torch.randn(32, 784)
    out = model(x)
    print(f"  Output: mean={out.data.mean():.4f}, std={out.data.std():.4f}")


# -----------------------------------------------------------
# 2. ALL PyTorch init functions at a glance
# -----------------------------------------------------------
print("\n\n" + "=" * 70)
print("ALL PyTorch nn.init FUNCTIONS")
print("=" * 70)

init_functions = {
    'nn.init.zeros_(tensor)': 'All zeros (use for biases only)',
    'nn.init.ones_(tensor)': 'All ones (use for BN gamma)',
    'nn.init.constant_(tensor, val)': 'Constant value',
    'nn.init.normal_(tensor, mean, std)': 'Random normal N(mean, std²)',
    'nn.init.uniform_(tensor, a, b)': 'Random uniform U(a, b)',
    'nn.init.xavier_normal_(tensor, gain)': 'Glorot Normal: N(0, 2/(fan_in+fan_out))',
    'nn.init.xavier_uniform_(tensor, gain)': 'Glorot Uniform: U(-a, a), a=sqrt(6/(fin+fout))',
    'nn.init.kaiming_normal_(tensor, a, mode, nonlinearity)': 'He Normal: N(0, 2/fan)',
    'nn.init.kaiming_uniform_(tensor, a, mode, nonlinearity)': 'He Uniform: U(-a, a)',
    'nn.init.orthogonal_(tensor, gain)': 'Orthogonal matrix (QR decomposition)',
    'nn.init.sparse_(tensor, sparsity, std)': 'Sparse initialization',
    'nn.init.eye_(tensor)': 'Identity matrix',
}

for func, desc in init_functions.items():
    print(f"  {func:<55} | {desc}")

print("\n\n✅ Notebook complete! Run all cells to see visualizations and comparisons.")